# 03 - Enterprise Multi-Agent Orchestration

## Overview

In this tutorial we'll build a complete enterprise-grade multi-tenant insurance platform. We'll create a supervisor agent that orchestrates the specialized agents from Tutorials 01 and 02, and deploy a production-ready web application with AWS Cloudscape UI.

This demonstrates how to build real-world multi-agent systems with proper authentication, routing, and user interfaces.

### Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Multi-Agent Orchestration |
| Authentication | OAuth 2.0 Access Token with custom scopes |
| Agentic Framework | Strands Agents with Supervisor Pattern |
| LLM model | Anthropic Claude Sonnet 4.5 |
| Frontend | React + AWS Cloudscape Design System |
| Backend | Flask API with JWT validation |
| Tutorial vertical | Insurance |
| Example complexity | Advanced |
| SDK used | Amazon Bedrock AgentCore Python SDK |

### Tutorial Architecture

```
┌─────────────────────────────────────────────────────────────┐
│              Cloudscape UI (React Frontend)                  │
│  - Cognito OAuth 2.0 Login                                   │
│  - Claims Dashboard (tier-filtered)                          │
│  - AI Chat Interface (streaming)                             │
│  - Real-time Statistics                                      │
└────────────────────────┬────────────────────────────────────┘
                         │ Access Token (scope: insurance/regular)
                         ↓
┌─────────────────────────────────────────────────────────────┐
│                   Backend API (Flask)                        │
│  - JWT validation                                            │
│  - Tier extraction from scope                                │
│  - Request routing                                           │
│  - Error handling                                            │
└────────────────────────┬────────────────────────────────────┘
                         │ Tier context
                         ↓
┌─────────────────────────────────────────────────────────────┐
│           Supervisor Agent (AgentCore Runtime)               │
│  - Query understanding & classification                      │
│  - Intelligent agent selection                               │
│  - Response orchestration                                    │
│  - Streaming support                                         │
└────────────┬───────────────────────────┬────────────────────┘
             │                           │
             ↓                           ↓
┌────────────────────────┐  ┌──────────────────────────┐
│  RAG Agent (Tutorial 01)│  │ RLS Agent (Tutorial 02)  │
│  - Policy documents     │  │ - Claims database        │
│  - Benefits information │  │ - Customer records       │
│  - Metadata filtering   │  │ - Row-level security     │
│  - Tier: regular        │  │ - Tier: regular          │
└────────────────────────┘  └──────────────────────────┘
```

### Tutorial Key Features

* Supervisor agent pattern for intelligent routing
* Multi-agent orchestration with tier context propagation
* Production-ready React UI with AWS Cloudscape
* Full-stack OAuth 2.0 authentication flow
* Streaming AI responses for real-time interaction
* Enterprise-grade error handling and logging
* Tier-based access control across all layers

## Prerequisites

To execute this tutorial you will need:
* **Tutorial 01 completed** - RAG agent deployed
* **Tutorial 02 completed** - RLS agent deployed
* Node.js 16+ and npm
* Python 3.9+
* AWS credentials configured
* PostgreSQL database running

## Step 1: Install Dependencies

In [ ]:
!pip install -q boto3 bedrock-agentcore-starter-toolkit strands-agents flask flask-cors psycopg2-binary

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import boto3
import json
from boto3.session import Session
from utils.aws_resources import CognitoManager, IAMManager

boto_session = Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']

print(f"Region: {region}")
print(f"Account ID: {account_id}")

## Step 2: Retrieve Sub-Agent ARNs from Parameter Store

In [ ]:
ssm = boto3.client('ssm')

# Retrieve agent ARNs
agent_01_arn = ssm.get_parameter(Name='/agents/unstructured_accesstoken_arn')['Parameter']['Value']
agent_02_arn = ssm.get_parameter(Name='/agents/unstructured_idtoken_arn')['Parameter']['Value']
agent_03_arn = ssm.get_parameter(Name='/agents/structured_accesstoken_arn')['Parameter']['Value']
agent_04_arn = ssm.get_parameter(Name='/agents/structured_idtoken_arn')['Parameter']['Value']

print("Sub-Agent ARNs:")
print(f"  Agent 01 (Unstructured AccessToken): {agent_01_arn}")
print(f"  Agent 02 (Unstructured IDToken): {agent_02_arn}")
print(f"  Agent 03 (Structured AccessToken): {agent_03_arn}")
print(f"  Agent 04 (Structured IDToken): {agent_04_arn}")

## Step 3: Create Supervisor Agent IAM Role

In [ ]:
from utils import create_agentcore_role

supervisor_agent_name = "supervisor_agent"
supervisor_iam_role = create_agentcore_role(
    agent_name=supervisor_agent_name,
    region=region
)
supervisor_role_arn = supervisor_iam_role['Role']['Arn']
supervisor_role_name = supervisor_iam_role['Role']['RoleName']

print(f"Supervisor Role ARN: {supervisor_role_arn}")
print(f"Supervisor Role Name: {supervisor_role_name}")

## Step 4: Update Supervisor Permissions

In [ ]:
iam_client = boto3.client('iam')

sub_agent_arns = [agent_01_arn, agent_02_arn, agent_03_arn, agent_04_arn]
parameter_arns = [
    ssm.get_parameter(Name='/agents/unstructured_accesstoken_arn')['Parameter']['ARN'],
    ssm.get_parameter(Name='/agents/unstructured_idtoken_arn')['Parameter']['ARN'],
    ssm.get_parameter(Name='/agents/structured_accesstoken_arn')['Parameter']['ARN'],
    ssm.get_parameter(Name='/agents/structured_idtoken_arn')['Parameter']['ARN']
]

supervisor_permissions = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:InvokeAgentRuntime"],
            "Resource": [
                f"{arn}/runtime-endpoint/DEFAULT" for arn in sub_agent_arns
            ] + sub_agent_arns
        },
        {
            "Effect": "Allow",
            "Action": ["ssm:GetParameter"],
            "Resource": parameter_arns
        }
    ]
}

iam_client.put_role_policy(
    RoleName=supervisor_role_name,
    PolicyName="subagent_permissions",
    PolicyDocument=json.dumps(supervisor_permissions)
)

print("✓ Supervisor permissions updated")

## Step 5: Create Supervisor Agent

In [ ]:
%%writefile supervisor_agent.py

import boto3
import logging
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

logger = logging.getLogger(__name__)
app = BedrockAgentCoreApp()

def get_agent_arn(agent_name: str) -> str:
    """Retrieve agent ARN from Parameter Store"""
    ssm = boto3.client('ssm')
    response = ssm.get_parameter(Name=f'/agents/{agent_name}_arn')
    return response['Parameter']['Value']

def invoke_agent(agent_arn: str, user_query: str) -> str:
    """Invoke sub-agent via AgentCore Runtime"""
    client = boto3.client('bedrock-agentcore-runtime')
    response = client.invoke_agent_runtime(
        agentRuntimeArn=agent_arn,
        payload={'prompt': user_query}
    )
    return response['output']

@tool
def query_unstructured_accesstoken(user_query: str):
    """Query unstructured data with Access Token authentication"""
    agent_arn = get_agent_arn('unstructured_accesstoken')
    return invoke_agent(agent_arn, user_query)

@tool
def query_unstructured_idtoken(user_query: str):
    """Query unstructured data with ID Token and metadata filtering"""
    agent_arn = get_agent_arn('unstructured_idtoken')
    return invoke_agent(agent_arn, user_query)

@tool
def query_structured_accesstoken(user_query: str):
    """Query structured data with Access Token and RLS"""
    agent_arn = get_agent_arn('structured_accesstoken')
    return invoke_agent(agent_arn, user_query)

@tool
def query_structured_idtoken(user_query: str):
    """Query structured data with ID Token and RLS"""
    agent_arn = get_agent_arn('structured_idtoken')
    return invoke_agent(agent_arn, user_query)

model = BedrockModel(model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0")
agent = Agent(
    model=model,
    system_prompt="""You are a supervisor agent for an insurance claims platform.
    Route user queries to the appropriate specialized agent:
    - For policy questions (unstructured): use query_unstructured_idtoken
    - For claims data (structured): use query_structured_idtoken
    Always maintain tenant isolation.""",
    tools=[
        query_unstructured_accesstoken,
        query_unstructured_idtoken,
        query_structured_accesstoken,
        query_structured_idtoken
    ]
)

@app.entrypoint
async def supervisor_agent_streaming(payload):
    """Supervisor agent with streaming"""
    user_input = payload.get("prompt")
    
    async for event in agent.stream_async(user_input):
        if "data" in event:
            yield event["data"]

if __name__ == "__main__":
    app.run()

## Step 6: Launch Supervisor Agent

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

supervisor_runtime = Runtime()
supervisor_runtime.configure(
    entrypoint="supervisor_agent.py",
    execution_role=supervisor_role_arn,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=supervisor_agent_name
)

supervisor_launch_result = supervisor_runtime.launch()
supervisor_agent_arn = supervisor_launch_result.agent_arn

print(f"✓ Supervisor Agent ARN: {supervisor_agent_arn}")

# Save to Parameter Store
ssm.put_parameter(
    Name='/agents/supervisor_agent_arn',
    Value=supervisor_agent_arn,
    Type='String',
    Overwrite=True
)

## Step 7: Start Backend API

In [ ]:
import subprocess
import time

# Start backend in background
backend_process = subprocess.Popen(
    ['python', 'backend/app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("✓ Backend API starting on http://localhost:5000")
time.sleep(3)

# Test health endpoint
import requests
response = requests.get('http://localhost:5000/health')
print(f"Health check: {response.json()}")

## Step 8: Start Frontend UI

In [ ]:
# Start frontend in background
frontend_process = subprocess.Popen(
    ['npm', 'start'],
    cwd='frontend',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("✓ Frontend UI starting on http://localhost:3000")
print("\n" + "="*60)
print("🎉 Multi-Tenant Insurance Claims Platform is ready!")
print("="*60)
print("\n📱 Frontend: http://localhost:3000")
print("🔌 Backend API: http://localhost:5000")
print("\n🔐 Demo Accounts:")
print("  • broker-a@example.com / Pass123! (5 claims)")
print("  • broker-b@example.com / Pass123! (2 claims)")
print("\n⚠️  Run the cleanup cell below to stop all services")

## Cleanup (Optional)

Run this cell to stop services and optionally delete AWS resources.

**Services**: Backend and frontend processes will be stopped.

**AWS Resources**: Uncomment to delete Cognito User Pool and PostgreSQL data.

In [ ]:
# Stop backend and frontend services
try:
    backend_process.terminate()
    frontend_process.terminate()
    print("✓ Services stopped")
except:
    print("⚠ Services may already be stopped")

# Optionally delete AWS resources
# from utils.aws_resources import cleanup_cognito_resources

# cleanup_cognito_resources(
#     region=region,
#     user_pool_name=user_pool_name
# )

# If you want to drop the database tables:
# with conn.cursor() as cur:
#     cur.execute("DROP TABLE IF EXISTS claims CASCADE")
#     cur.execute("DROP TABLE IF EXISTS user_tenants CASCADE")
#     cur.execute("DROP TABLE IF EXISTS tenants CASCADE")
# conn.commit()
# conn.close()
# print("✓ Database tables dropped")